<br>

# Municípios

Explorando o _site_ [ListaTelefonica](https://www.tjsp.jus.br/ListaTelefonica) foi possível encontrar APIs que operam e fazer diversas pesquisas.


In [ ]:
import os

import pandas as pd
from dotenv import load_dotenv

from tjsp_unidades import api
from tjsp_unidades.paths import output_path_tab

<br>

---

## Lista de Municípios


Criamos função que pega o nome e Código do Município, de acordo com o TJSP.


In [ ]:
listar = api.ListarMunicipios()
listar.get_lista_municipios_tjsp(termo='San')

Para termos um _input_ para a função, que é necessário inserir parte do nome, usamos os nomes dos municípios.


In [ ]:
listar.lista_municipios[0:5]

<br>

Obtive o município com maior número de caracteres, visto que a função do método `POST` sempre lista apenas 10 municípios...


In [ ]:
listar.n_caracteres_mun_max

<br>

Uma vez com a função que retorna os municípios escrita, fiz uma iteração por todos os trechos de nomes de municípios. Para cada municípios foi pesquisado diversas vezes, incrementando o número de caracteres. Por exemplos, o município de Santos foi pesquisado:

- San
- Sant
- Santo
- Santos

<br>

Isso garantiu que todos os municípios foram "raspados".


In [ ]:
listar.list_termos[0:5]

<br>

Descobri uma maneira de paralelizar as consultas.\
Antes eu levava cerca de 40 minutos para rodar. Agora consigo fazer com 7 minutos.


In [ ]:
df_tjsp = listar.request()

# Results
df_tjsp.info()
df_tjsp.head()

<br>

Contudo, é possível ainda deixar mais rápido, utilizando a API da AWS.


In [ ]:
# load_dotenv()

# AWS_ACCESS_KEY_ID = os.getenv('AWS_ACCESS_KEY_ID')
# AWS_SECRET_ACCESS_KEY = os.getenv('AWS_SECRET_ACCESS_KEY')

In [ ]:
# listar.request_aws(
#     access_key_id=AWS_ACCESS_KEY_ID,
#     access_key_secret=AWS_SECRET_ACCESS_KEY,
# )

Ajusto a tabela criada


In [ ]:
# # Ajusta a tabela
# df_tjsp = listar.transform_municipios_tjsp()

# # Results
# df_tjsp.info()
# df_tjsp.head()

In [ ]:
filename = "Municipios"

# Salva
df_tjsp.to_csv(
    path_or_buf=output_path_tab / f"{filename}.csv",
    index=False,
)

# df_tjsp.to_excel(
#     excel_writer=output_path_tab / f"{filename}.xlsx",
#     sheet_name=f"{filename}",
#     index=False,
# )

<br>

---

## Análise dos Nomes dos Municípios

A ideia principal é agregar os códigos do IBGE na tabela. Para isso eu fiz um _join_... vi os erros... e corrigi!


In [ ]:
# Read Data
df_tjsp = pd.read_csv(filepath_or_buffer=output_path_tab / "Municipios.csv")

# Results
df_tjsp.info()
df_tjsp.head()

<br>

Peguei a tabela padrão, que fiz!, no projeto do [open-geodata](https://github.com/michelmetran/open-geodata)!


In [ ]:
mun = api.MunicipiosTJSP(df_municipios=df_tjsp)
mun.nomes_corretos

<br>

Juntei


In [ ]:
mun.agrega_nomes_corretos(
    dict_replace={
        # Nome Errado (TJSP): Nome Correto
        "Estrela dOeste": "Estrela d'Oeste",
        "Luís Antônio": "Luiz Antônio",
        "Florínia": "Florínea",
    }
)

In [ ]:
print(list(mun.df_municipios.columns))
mun.df_municipios

<br>

Uma vez que sei os erros dos nomes e são apenas 3 registros...
Criei uma coluna duplicada para corrigir!


In [ ]:
# Checa se foi substituido na coluna "Nome Corrigido"
mun.municipios[mun.municipios["municipio_tjsp_corrigido"].str.startswith("Florí")]

<br>

Por fim, checo novamente e vejo que não tem mais nada nulo! Excelente!


In [ ]:
filename = "Municipios"

# Salva
mun.df_municipios.to_csv(
    path_or_buf=output_path_tab / f"{filename}.csv",
    index=False,
)
# mun.df_municipios.to_excel(
#     excel_writer=output_path_tab / f"{filename}.xlsx",
#     sheet_name=f"{filename}",
#     index=False,
# )